
# \(d=3,\;Y4\) SU(3) vertex-singlet dictionary generator — fixed

This replacement is **self-contained**. It does not clone, install, or import
`pyclebsch`, so it is not vulnerable to Git, PIP, package-version, or runtime
restart failures.

It implements exact SU(3) tensor products with the Littlewood–Richardson rule,
then generates the singlet multiplicity for all

\[
10^6=1{,}000{,}000
\]

ordered six-link configurations at a cubic-lattice vertex.

Use a **standard Colab CPU runtime** and select **Runtime → Run all**.

Outputs are written under:

```text
/content/Y4_D3_VERTEX/
```

The final cell also creates:

```text
/content/Y4_D3_VERTEX/y4_d3_vertex_outputs.zip
```


In [1]:

from __future__ import annotations

import gzip
import itertools
import json
import math
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
OUT = BASE / "Y4_D3_VERTEX"
OUT.mkdir(parents=True, exist_ok=True)

IRREPS_DYNKIN = [
    (0, 0),  # 1
    (1, 0),  # 3
    (0, 1),  # 3bar
    (2, 0),  # 6
    (0, 2),  # 6bar
    (1, 1),  # 8
    (3, 0),  # 10
    (0, 3),  # 10bar
    (2, 1),  # 15
    (1, 2),  # 15bar
]
IRREP_NAMES = [
    "1", "3", "3bar", "6", "6bar", "8",
    "10", "10bar", "15", "15bar",
]

def dynkin_to_iweight(pq):
    p, q = map(int, pq)
    return (p + q, q, 0)

IRREPS_IWEIGHT = [dynkin_to_iweight(x) for x in IRREPS_DYNKIN]
IRREP_INDEX = {irrep: i for i, irrep in enumerate(IRREPS_DYNKIN)}

TRIPLES = list(itertools.product(range(10), repeat=3))
assert len(TRIPLES) == 1000

def encode_triple(indices):
    a, b, c = map(int, indices)
    return 100 * a + 10 * b + c

def decode_triple(code):
    code = int(code)
    return (code // 100, (code // 10) % 10, code % 10)

for index, triple in enumerate(TRIPLES):
    assert encode_triple(triple) == index
    assert decode_triple(index) == triple

print("Output directory:", OUT)
print("Irreps:", dict(enumerate(IRREP_NAMES)))


Output directory: /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX
Irreps: {0: '1', 1: '3', 2: '3bar', 3: '6', 4: '6bar', 5: '8', 6: '10', 7: '10bar', 8: '15', 9: '15bar'}


In [2]:

# ---------------------------------------------------------------------------
# Exact SU(3) Littlewood–Richardson engine
# ---------------------------------------------------------------------------

def su3_dimension(pq):
    p, q = map(int, pq)
    return (p + 1) * (q + 1) * (p + q + 2) // 2

def conjugate_dynkin(pq):
    p, q = map(int, pq)
    return (q, p)

def dynkin_to_partition(pq):
    p, q = map(int, pq)
    return (p + q, q, 0)

def partition_to_dynkin(partition):
    a, b, c = map(int, partition)
    return (a - b, b - c)

def partitions_at_most_three_rows(total):
    for a in range(total, -1, -1):
        for b in range(min(a, total - a), -1, -1):
            c = total - a - b
            if b >= c >= 0:
                yield (a, b, c)

@lru_cache(maxsize=None)
def lr_coefficient(lam, mu, nu):
    """
    Littlewood–Richardson coefficient c^nu_{lam,mu}.

    The skew diagram nu/lam is filled in reading order:
    top row to bottom row, right to left.
    """
    lam = tuple(map(int, lam))
    mu = tuple(int(x) for x in mu if int(x) > 0)
    nu = tuple(map(int, nu))

    if sum(nu) != sum(lam) + sum(mu):
        return 0
    if any(lam[i] > nu[i] for i in range(3)):
        return 0

    boxes = [
        (row, col)
        for row in range(3)
        for col in range(nu[row], lam[row], -1)
    ]
    if len(boxes) != sum(mu):
        return 0

    remaining = list(mu)
    prefix_counts = [0] * len(mu)
    assigned = {}
    count = 0

    def recurse(position):
        nonlocal count
        if position == len(boxes):
            count += 1
            return

        row, col = boxes[position]
        right = (row, col + 1)
        above = (row - 1, col)

        for value in range(1, len(mu) + 1):
            idx = value - 1
            if remaining[idx] == 0:
                continue

            # Rows are weakly increasing from left to right.
            if right in assigned and value > assigned[right]:
                continue

            # Columns are strictly increasing from top to bottom.
            if above in assigned and assigned[above] >= value:
                continue

            remaining[idx] -= 1
            prefix_counts[idx] += 1

            # Lattice/Yamanouchi word condition.
            valid = all(
                prefix_counts[j] >= prefix_counts[j + 1]
                for j in range(len(prefix_counts) - 1)
            )

            if valid:
                assigned[(row, col)] = value
                recurse(position + 1)
                del assigned[(row, col)]

            prefix_counts[idx] -= 1
            remaining[idx] += 1

    recurse(0)
    return count

@lru_cache(maxsize=None)
def su3_tensor_product(left, right):
    left = tuple(map(int, left))
    right = tuple(map(int, right))

    lam = dynkin_to_partition(left)
    mu = dynkin_to_partition(right)
    total = sum(lam) + sum(mu)

    result = Counter()
    for nu in partitions_at_most_three_rows(total):
        if any(nu[i] < lam[i] for i in range(3)):
            continue
        multiplicity = lr_coefficient(lam, mu, nu)
        if multiplicity:
            result[partition_to_dynkin(nu)] += multiplicity

    return dict(result)

def su3_multi_product(representations):
    result = {(0, 0): 1}
    for representation in representations:
        next_result = Counter()
        for left, left_mult in result.items():
            for right, right_mult in su3_tensor_product(
                left, tuple(representation)
            ).items():
                next_result[right] += left_mult * right_mult
        result = dict(next_result)
    return result

# Basic exact anchors.
assert su3_tensor_product((1, 0), (1, 0)) == {
    (2, 0): 1, (0, 1): 1
}
assert su3_tensor_product((1, 0), (0, 1)) == {
    (1, 1): 1, (0, 0): 1
}
assert su3_tensor_product((1, 1), (1, 0)) == {
    (2, 1): 1, (0, 2): 1, (1, 0): 1
}
assert su3_tensor_product((1, 1), (0, 1)) == {
    (1, 2): 1, (2, 0): 1, (0, 1): 1
}

print("LR ENGINE PASS: exact SU(3) tensor-product anchors")


LR ENGINE PASS: exact SU(3) tensor-product anchors


## Verify the certified 30-edge skeleton

In [3]:

CERTIFIED_EDGES = json.loads('[{"edge_id":"FTI-01","source_dynkin":[0,0],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-02","source_dynkin":[0,0],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-03","source_dynkin":[0,1],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-04","source_dynkin":[0,1],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-05","source_dynkin":[0,1],"token":1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-06","source_dynkin":[0,1],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-07","source_dynkin":[0,2],"token":-1,"target_dynkin":[0,3],"target_dimension":10},{"edge_id":"FTI-08","source_dynkin":[0,2],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-09","source_dynkin":[0,2],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-10","source_dynkin":[0,2],"token":1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-11","source_dynkin":[0,3],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-12","source_dynkin":[1,0],"token":-1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-13","source_dynkin":[1,0],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-14","source_dynkin":[1,0],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-15","source_dynkin":[1,0],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-16","source_dynkin":[1,1],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-17","source_dynkin":[1,1],"token":-1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-18","source_dynkin":[1,1],"token":-1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-19","source_dynkin":[1,1],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-20","source_dynkin":[1,1],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-21","source_dynkin":[1,1],"token":1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-22","source_dynkin":[1,2],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-23","source_dynkin":[1,2],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-24","source_dynkin":[2,0],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-25","source_dynkin":[2,0],"token":-1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-26","source_dynkin":[2,0],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-27","source_dynkin":[2,0],"token":1,"target_dynkin":[3,0],"target_dimension":10},{"edge_id":"FTI-28","source_dynkin":[2,1],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-29","source_dynkin":[2,1],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-30","source_dynkin":[3,0],"token":-1,"target_dynkin":[2,0],"target_dimension":6}]')

edge_results = []
for row in CERTIFIED_EDGES:
    source = tuple(row["source_dynkin"])
    token = (1, 0) if int(row["token"]) == 1 else (0, 1)
    target = tuple(row["target_dynkin"])

    decomposition = su3_tensor_product(source, token)
    multiplicity = int(decomposition.get(target, 0))
    dimension = su3_dimension(target)

    record = {
        "edge_id": row["edge_id"],
        "source_dynkin": list(source),
        "token": int(row["token"]),
        "target_dynkin": list(target),
        "multiplicity": multiplicity,
        "dimension": dimension,
        "passed": (
            multiplicity == 1
            and dimension == int(row["target_dimension"])
        ),
    }
    edge_results.append(record)

failed = [row for row in edge_results if not row["passed"]]
assert not failed, failed
assert len(edge_results) == 30

print("EDGE GATE PASS: 30/30 certified fusion edges")


EDGE GATE PASS: 30/30 certified fusion edges


## Generate all ordered triple decompositions

In [4]:

# ---------------------------------------------------------------------------
# All 1000 ordered triple decompositions
# ---------------------------------------------------------------------------

started = time.time()
triple_decompositions = []

for index, triple in enumerate(TRIPLES):
    first, second, third = (
        IRREPS_DYNKIN[triple[0]],
        IRREPS_DYNKIN[triple[1]],
        IRREPS_DYNKIN[triple[2]],
    )
    decomposition = su3_multi_product((first, second, third))

    lhs_dimension = (
        su3_dimension(first)
        * su3_dimension(second)
        * su3_dimension(third)
    )
    rhs_dimension = sum(
        multiplicity * su3_dimension(irrep)
        for irrep, multiplicity in decomposition.items()
    )
    assert lhs_dimension == rhs_dimension, {
        "triple": triple,
        "lhs": lhs_dimension,
        "rhs": rhs_dimension,
    }

    triple_decompositions.append(decomposition)

assert len(triple_decompositions) == 1000

cache_path = OUT / "y4_triple_decompositions.json.gz"
serializable = [
    {
        f"{irrep[0]},{irrep[1]}": int(multiplicity)
        for irrep, multiplicity in sorted(decomposition.items())
    }
    for decomposition in triple_decompositions
]
with gzip.GzipFile(
    filename=str(cache_path),
    mode="wb",
    compresslevel=9,
    mtime=0,
) as handle:
    handle.write(
        json.dumps(
            serializable,
            separators=(",", ":"),
            sort_keys=True,
        ).encode("utf-8")
    )

print(
    "TRIPLE GATE PASS: 1000/1000 decompositions "
    f"in {time.time() - started:.3f} s"
)


TRIPLE GATE PASS: 1000/1000 decompositions in 0.072 s


## Build the six-link singlet table

In [5]:

# ---------------------------------------------------------------------------
# Million-entry six-link singlet multiplicity table
# ---------------------------------------------------------------------------

all_channels = sorted({
    irrep
    for decomposition in triple_decompositions
    for irrep in decomposition
})
channel_index = {irrep: index for index, irrep in enumerate(all_channels)}

# int64 is intentional: it eliminates any possibility of overflow.
channel_matrix = np.zeros(
    (1000, len(all_channels)),
    dtype=np.int64,
)
for triple_index, decomposition in enumerate(triple_decompositions):
    for irrep, multiplicity in decomposition.items():
        channel_matrix[
            triple_index,
            channel_index[irrep],
        ] = int(multiplicity)

multiplicities = channel_matrix @ channel_matrix.T

assert multiplicities.shape == (1000, 1000)
assert np.all(multiplicities >= 0)

nonzero_assignments = int(np.count_nonzero(multiplicities))
maximum_multiplicity = int(multiplicities.max())
multiplicity_histogram = Counter(
    map(int, multiplicities.ravel())
)

print("MATRIX GATE PASS")
print("  assignments             :", multiplicities.size)
print("  nonzero singlet sectors :", nonzero_assignments)
print("  maximum multiplicity    :", maximum_multiplicity)
print("  intermediate channels   :", len(all_channels))


MATRIX GATE PASS
  assignments             : 1000000
  nonzero singlet sectors : 330286
  maximum multiplicity    : 798
  intermediate channels   : 55


## Direct six-factor validation

In [6]:

# ---------------------------------------------------------------------------
# Independent direct six-factor product checks
# ---------------------------------------------------------------------------

anchor_pairs = [
    (0, 0),
    (111, 111),
    (123, 456),
    (555, 555),
    (678, 876),
    (999, 999),
    (5, 500),
    (321, 123),
    (808, 80),
    (246, 642),
]

anchor_results = []
for outgoing_code, incoming_code in anchor_pairs:
    outgoing_indices = decode_triple(outgoing_code)
    incoming_indices = decode_triple(incoming_code)

    direct_factors = [
        IRREPS_DYNKIN[index]
        for index in outgoing_indices
    ] + [
        conjugate_dynkin(IRREPS_DYNKIN[index])
        for index in incoming_indices
    ]

    direct_singlet_multiplicity = int(
        su3_multi_product(direct_factors).get((0, 0), 0)
    )
    factorized_multiplicity = int(
        multiplicities[outgoing_code, incoming_code]
    )

    record = {
        "outgoing_code": outgoing_code,
        "incoming_code": incoming_code,
        "direct": direct_singlet_multiplicity,
        "factorized": factorized_multiplicity,
        "passed": (
            direct_singlet_multiplicity
            == factorized_multiplicity
        ),
    }
    anchor_results.append(record)
    assert record["passed"], record

print(
    "SIX-FOLD GATE PASS:",
    len(anchor_results),
    "independent anchors",
)


SIX-FOLD GATE PASS: 10 independent anchors


## Write and package the outputs

In [7]:

# ---------------------------------------------------------------------------
# Write and round-trip all outputs
# ---------------------------------------------------------------------------

npz_path = OUT / "y4_d3_vertex_multiplicity_table.npz"
np.savez_compressed(
    npz_path,
    multiplicities=multiplicities,
    triple_indices=np.asarray(TRIPLES, dtype=np.uint8),
    irrep_dynkin=np.asarray(IRREPS_DYNKIN, dtype=np.int16),
    irrep_iweights=np.asarray(IRREPS_IWEIGHT, dtype=np.int16),
    channel_dynkin=np.asarray(all_channels, dtype=np.int16),
)

summary = {
    "version": "2026-06-13-d3-y4-vertex-v2-fixed",
    "method": "self-contained exact SU(3) Littlewood-Richardson decomposition",
    "external_package_dependency": False,
    "orientation_convention": {
        "outgoing": ["+x", "+y", "+z"],
        "incoming": ["-x", "-y", "-z"],
        "invariant_space": (
            "Inv(R_out1 tensor R_out2 tensor R_out3 tensor "
            "conj(R_in1) tensor conj(R_in2) tensor conj(R_in3))"
        ),
    },
    "irreps": [
        {
            "index": index,
            "name": IRREP_NAMES[index],
            "dynkin": list(IRREPS_DYNKIN[index]),
            "iweight": list(IRREPS_IWEIGHT[index]),
            "dimension": su3_dimension(IRREPS_DYNKIN[index]),
        }
        for index in range(10)
    ],
    "ordered_triples": 1000,
    "ordered_six_link_assignments": int(multiplicities.size),
    "nonzero_singlet_assignments": nonzero_assignments,
    "maximum_singlet_multiplicity": maximum_multiplicity,
    "multiplicity_histogram": {
        str(key): int(value)
        for key, value in sorted(multiplicity_histogram.items())
    },
    "intermediate_irrep_channels": len(all_channels),
    "certified_fusion_edges": edge_results,
    "six_factor_anchors": anchor_results,
    "files": {
        "multiplicity_table": npz_path.name,
        "triple_decompositions": cache_path.name,
    },
    "passed": True,
}

summary_path = OUT / "y4_d3_vertex_multiplicity_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

lookup_source = """from pathlib import Path
import numpy as np

_ROOT = Path(__file__).resolve().parent
_DATA = np.load(_ROOT / "y4_d3_vertex_multiplicity_table.npz")

MULTIPLICITIES = _DATA["multiplicities"]
IRREPS_DYNKIN = [
    tuple(map(int, row))
    for row in _DATA["irrep_dynkin"]
]
IRREP_INDEX = {
    irrep: index
    for index, irrep in enumerate(IRREPS_DYNKIN)
}

def encode_triple(indices):
    a, b, c = map(int, indices)
    return 100 * a + 10 * b + c

def singlet_multiplicity(outgoing_indices, incoming_indices):
    return int(MULTIPLICITIES[
        encode_triple(outgoing_indices),
        encode_triple(incoming_indices),
    ])

def singlet_multiplicity_dynkin(outgoing, incoming):
    return singlet_multiplicity(
        [IRREP_INDEX[tuple(x)] for x in outgoing],
        [IRREP_INDEX[tuple(x)] for x in incoming],
    )
"""

lookup_path = OUT / "y4_d3_vertex_lookup.py"
lookup_path.write_text(lookup_source, encoding="utf-8")

loaded = np.load(npz_path)
assert np.array_equal(
    loaded["multiplicities"],
    multiplicities,
)
assert json.loads(
    summary_path.read_text(encoding="utf-8")
)["passed"] is True

import zipfile

zip_path = OUT / "y4_d3_vertex_outputs.zip"
with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in (
        npz_path,
        summary_path,
        lookup_path,
        cache_path,
    ):
        archive.write(path, arcname=path.name)

print("OUTPUT GATE PASS")
print("NPZ    :", npz_path)
print("SUMMARY:", summary_path)
print("LOOKUP :", lookup_path)
print("CACHE  :", cache_path)
print("ZIP    :", zip_path)
print("ALL FIXED-NOTEBOOK GATES PASS")


OUTPUT GATE PASS
NPZ    : /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX/y4_d3_vertex_multiplicity_table.npz
SUMMARY: /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX/y4_d3_vertex_multiplicity_summary.json
LOOKUP : /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX/y4_d3_vertex_lookup.py
CACHE  : /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX/y4_triple_decompositions.json.gz
ZIP    : /mnt/data/y4_vertex_fixed_execution/Y4_D3_VERTEX/y4_d3_vertex_outputs.zip
ALL FIXED-NOTEBOOK GATES PASS
